# ML Dataset v4

Create `ml_data_v4` folder structure and copy QC-referenced PNG + NPY from dent, girth_weld, and girth_weld_neg (random) sources for each inspection_id.

**False positives:** copied from `qc_results.json` **without** `approved`/`edited` filter. For **0ABP0TFUSH1**, rows are filtered by PNG basename in `.../std_normalized/qc/qced/`; **whitelisted PNGs are copied from `qc/qced`** (std and patch if `patch_normalized/qc/qced` exists), NPY still from JSON paths.

**Noise / scale augmentation** for training positives lives in **`data_preparation/augmentation/augment_v4.ipynb`** (after roll aug on `pos/`). Do not duplicate here.

In [32]:
import os
import shutil
from pathlib import Path

# Base paths (Linux; Windows UNC \\wsl...\home\... -> /home/...)
BASE = Path("/home/zmirikha/Github/rnd_q2/data")
OUT_ROOT = Path("/home/zmirikha/Github/rnd_q2/data/ml_data_v4")

# Inspection IDs to support
# Refs: data/{id}/girth_weld/std_normalized/qc, girth_weld/patch_normalized/qc,
#       data/{id}/girth_weld_neg/std_normalized/qc, girth_weld_neg/patch_normalized/qc
INSPECTION_IDS = ["0A1OLRV96N3", "0A49KLT3B7Y", "0AGBXB4WEGN", "0ABP0TFUSH1"]

# Reference QC folders per category (parent of 'qc' is where .npy lives)
# Format: (category_key, ref_relative_path)
# ref_relative_path is under BASE / inspection_id (e.g. "girth_weld_neg/std_normalized")
# Dent copy disabled; only girth_weld and random (girth_weld_neg) are copied.
REFERENCE_SOURCES = [
    ("random", "girth_weld_neg"),   # girth_weld_neg -> random
    ("girth_weld", "girth_weld"),
]

In [33]:
# Create ml_data_v4 structure: {dent, girth_weld, random} / {inspection_id} / {std_normalized, patch_normalized} / qc
for category in ["dent", "girth_weld", "random", "fp"]:
    for iid in INSPECTION_IDS:
        for norm in ["std_normalized", "patch_normalized"]:
            qc_dir = OUT_ROOT / category / iid / norm / "qc"
            (OUT_ROOT / category / iid / norm).mkdir(parents=True, exist_ok=True)
            qc_dir.mkdir(parents=True, exist_ok=True)
print(f"Created folder structure under {OUT_ROOT}")
for c in ["dent", "girth_weld", "random"]:
    print(f"  {c}/")
    for iid in INSPECTION_IDS:
        print(f"    {iid}/std_normalized/qc, {iid}/patch_normalized/qc")

Created folder structure under /home/zmirikha/Github/rnd_q2/data/ml_data_v4
  dent/
    0A1OLRV96N3/std_normalized/qc, 0A1OLRV96N3/patch_normalized/qc
    0A49KLT3B7Y/std_normalized/qc, 0A49KLT3B7Y/patch_normalized/qc
    0AGBXB4WEGN/std_normalized/qc, 0AGBXB4WEGN/patch_normalized/qc
    0ABP0TFUSH1/std_normalized/qc, 0ABP0TFUSH1/patch_normalized/qc
  girth_weld/
    0A1OLRV96N3/std_normalized/qc, 0A1OLRV96N3/patch_normalized/qc
    0A49KLT3B7Y/std_normalized/qc, 0A49KLT3B7Y/patch_normalized/qc
    0AGBXB4WEGN/std_normalized/qc, 0AGBXB4WEGN/patch_normalized/qc
    0ABP0TFUSH1/std_normalized/qc, 0ABP0TFUSH1/patch_normalized/qc
  random/
    0A1OLRV96N3/std_normalized/qc, 0A1OLRV96N3/patch_normalized/qc
    0A49KLT3B7Y/std_normalized/qc, 0A49KLT3B7Y/patch_normalized/qc
    0AGBXB4WEGN/std_normalized/qc, 0AGBXB4WEGN/patch_normalized/qc
    0ABP0TFUSH1/std_normalized/qc, 0ABP0TFUSH1/patch_normalized/qc


In [34]:
def npy_name_for_png(png_name: str) -> str:
    """Infer NPY filename from PNG (e.g. 0000922910_015.png -> d0000922910_015.npy)."""
    stem = Path(png_name).stem
    return f"d{stem}.npy"

def copy_qc_refs_balanced(
    ref_base: Path,
    out_category_dir: Path,
) -> tuple[int, int]:
    """Copy only samples that exist in BOTH std and patch refs so both output folders have equal data."""
    std_qc = ref_base / "std_normalized" / "qc"
    patch_qc = ref_base / "patch_normalized" / "qc"
    std_parent = ref_base / "std_normalized"
    patch_parent = ref_base / "patch_normalized"
    
    if not std_qc.exists() or not patch_qc.exists():
        return 0, 0
    
    std_pngs = {p.name for p in std_qc.glob("*.png")}
    patch_pngs = {p.name for p in patch_qc.glob("*.png")}
    common = std_pngs & patch_pngs
    
    out_std_qc = out_category_dir / "std_normalized" / "qc"
    out_std_npy = out_category_dir / "std_normalized"
    out_patch_qc = out_category_dir / "patch_normalized" / "qc"
    out_patch_npy = out_category_dir / "patch_normalized"
    for d in (out_std_qc, out_std_npy, out_patch_qc, out_patch_npy):
        d.mkdir(parents=True, exist_ok=True)
    
    copied = 0
    for png_name in sorted(common):
        npy_name = npy_name_for_png(png_name)
        src_std_png = std_qc / png_name
        src_std_npy = std_parent / npy_name
        src_patch_png = patch_qc / png_name
        src_patch_npy = patch_parent / npy_name
        if not (src_std_png.exists() and src_std_npy.exists() and src_patch_png.exists() and src_patch_npy.exists()):
            continue
        shutil.copy2(src_std_png, out_std_qc / png_name)
        shutil.copy2(src_std_npy, out_std_npy / npy_name)
        shutil.copy2(src_patch_png, out_patch_qc / png_name)
        shutil.copy2(src_patch_npy, out_patch_npy / npy_name)
        copied += 1
    return copied, len(common)

In [35]:
# Copy from reference folders: only samples present in BOTH std and patch are copied,
# so std_normalized and patch_normalized have equal data per category/inspection_id.

for iid in INSPECTION_IDS:
    for category, ref_subdir in REFERENCE_SOURCES:
        ref_base = BASE / iid / ref_subdir
        out_cat = OUT_ROOT / category / iid
        copied, common = copy_qc_refs_balanced(ref_base, out_cat)
        if common == 0:
            print(f"Skip {category}/{iid}: no ref folders or no common samples")
        else:
            print(f"{category}/{iid}: copied {copied} samples (same in std + patch); {common} in intersection")

random/0A1OLRV96N3: copied 135 samples (same in std + patch); 135 in intersection
girth_weld/0A1OLRV96N3: copied 500 samples (same in std + patch); 500 in intersection
random/0A49KLT3B7Y: copied 257 samples (same in std + patch); 257 in intersection
girth_weld/0A49KLT3B7Y: copied 500 samples (same in std + patch); 500 in intersection
random/0AGBXB4WEGN: copied 698 samples (same in std + patch); 698 in intersection
girth_weld/0AGBXB4WEGN: copied 995 samples (same in std + patch); 995 in intersection
random/0ABP0TFUSH1: copied 224 samples (same in std + patch); 224 in intersection
girth_weld/0ABP0TFUSH1: copied 1000 samples (same in std + patch); 1000 in intersection


In [41]:
import json
import re

QC_RESULTS_DIR = BASE.parent / "data_preparation" / "qc_results"
DENT_REF_SUBDIRS = ("Approved_qc", "Approved", "QC_Size_Size_Anomaly_Approved")  # try each; QC folder name varies

def _sanitize_qc_subdir(path: Path) -> str:
    """Same as app: safe subfolder name from path for qc_results."""
    s = path.resolve().as_posix().strip("/")
    s = re.sub(r"[^\w\-.]", "_", s)
    return s[:80] if len(s) > 80 else s or "default"

def copy_dents_from_qc_json(iid: str) -> int:
    """Copy dent samples with status approved/edited from QC results into ml_data_v4/dent/{iid}. Filename = {sample_id}_{final_labels}."""
    qc_json_path = None
    for ref_subdir in DENT_REF_SUBDIRS:
        ref_path = BASE / iid / ref_subdir
        qc_subdir = _sanitize_qc_subdir(ref_path)
        candidate = QC_RESULTS_DIR / qc_subdir / "qc_results.json"
        if candidate.exists():
            qc_json_path = candidate
            break
    if qc_json_path is None:
        print(f"Dent QC not found for {iid} (tried {DENT_REF_SUBDIRS})")
        return 0
    with open(qc_json_path, "r") as f:
        qc_data = json.load(f)
    out_dent = OUT_ROOT / "dent" / iid
    (out_dent / "std_normalized" / "qc").mkdir(parents=True, exist_ok=True)
    (out_dent / "std_normalized").mkdir(parents=True, exist_ok=True)
    (out_dent / "patch_normalized" / "qc").mkdir(parents=True, exist_ok=True)
    (out_dent / "patch_normalized").mkdir(parents=True, exist_ok=True)
    copied = 0
    for sample_id, data in qc_data.items():
        if data.get("status") not in ("edited", "approved"):
            continue
        final_labels = data.get("final_labels") or data.get("raw_labels") or []
        labels_str = str(final_labels).replace(" ", "").replace('"', "'")
        basename = f"{sample_id}_{labels_str}"
        png_name = basename + ".png"
        npy_name = "d" + basename + ".npy"
        for norm, path_key_png, path_key_npy in [
            ("std_normalized", "png_std_path", "npy_std_path"),
            ("patch_normalized", "png_patch_path", "npy_patch_path"),
        ]:
            src_png = Path(data[path_key_png]) if data.get(path_key_png) else None
            src_npy = Path(data[path_key_npy]) if data.get(path_key_npy) else None
            if src_png and src_png.exists():
                shutil.copy2(src_png, out_dent / norm / "qc" / png_name)
            if src_npy and src_npy.exists():
                shutil.copy2(src_npy, out_dent / norm / npy_name)
        copied += 1
    return copied

FP_REF_SUBDIRS = ("False_Positive", "False Positive")

# Whitelist: .../False Positive/std_normalized/qc/qced/*.png
# For whitelisted rows, PNGs are read from qc/qced (not from JSON paths).
def _fp_whitelist_and_qced_dirs(
    iid: str,
) -> tuple[frozenset[str] | None, Path | None, Path | None]:
    """Return (allowed_png_basenames, std_qced_dir, patch_qced_dir). (None,None,None) = no whitelist."""
    if iid != "0ABP0TFUSH1":
        return None, None, None
    for fp_folder in ("False Positive", "False_Positive"):
        root_fp = BASE / iid / fp_folder
        std_qced = root_fp / "std_normalized" / "qc" / "qced"
        patch_qced = root_fp / "patch_normalized" / "qc" / "qced"
        if std_qced.is_dir():
            names = {p.name for p in std_qced.glob("*.png")}
            pq = patch_qced if patch_qced.is_dir() else None
            print(f"FP whitelist: {std_qced} ({len(names)} PNGs); patch qced: {pq}")
            return frozenset(names), std_qced, pq
    print(
        f"FP whitelist folder not found for {iid} (tried 'False Positive' / std_normalized/qc/qced); no FP copied."
    )
    return frozenset(), None, None


def copy_fp_from_qc_json(iid: str) -> int:
    """Copy FP from qc_results into ml_data_v4/fp/{iid}. 0ABP0TFUSH1: whitelist from std qced; PNG src = qc/qced, NPY from JSON."""
    qc_json_path = None
    for ref_subdir in FP_REF_SUBDIRS:
        ref_path = BASE / iid / ref_subdir
        qc_subdir = _sanitize_qc_subdir(ref_path)
        candidate = QC_RESULTS_DIR / qc_subdir / "qc_results.json"
        if candidate.exists():
            qc_json_path = candidate
            break
    if qc_json_path is None:
        return 0
    with open(qc_json_path, "r") as f:
        qc_data = json.load(f)
    allowed_pngs, qced_std_dir, qced_patch_dir = _fp_whitelist_and_qced_dirs(iid)
    if allowed_pngs is not None and len(allowed_pngs) == 0:
        return 0
    out_fp = OUT_ROOT / "fp" / iid
    (out_fp / "std_normalized" / "qc").mkdir(parents=True, exist_ok=True)
    (out_fp / "std_normalized").mkdir(parents=True, exist_ok=True)
    (out_fp / "patch_normalized" / "qc").mkdir(parents=True, exist_ok=True)
    (out_fp / "patch_normalized").mkdir(parents=True, exist_ok=True)
    copied = 0
    skipped = 0
    for sample_id, data in qc_data.items():
        final_labels = data.get("final_labels") or data.get("raw_labels") or []
        labels_str = str(final_labels).replace(" ", "").replace('"', "'")
        basename = f"{sample_id}_{labels_str}"
        png_name = basename + ".png"
        npy_name = "d" + basename + ".npy"
        if allowed_pngs is not None and png_name not in allowed_pngs:
            skipped += 1
            continue
        for norm, path_key_png, path_key_npy in [
            ("std_normalized", "png_std_path", "npy_std_path"),
            ("patch_normalized", "png_patch_path", "npy_patch_path"),
        ]:
            src_npy = Path(data[path_key_npy]) if data.get(path_key_npy) else None
            use_qced = allowed_pngs is not None and png_name in allowed_pngs
            if use_qced and norm == "std_normalized" and qced_std_dir is not None:
                qced_png = qced_std_dir / png_name
                if qced_png.is_file():
                    shutil.copy2(qced_png, out_fp / norm / "qc" / png_name)
                else:
                    src_png = Path(data[path_key_png]) if data.get(path_key_png) else None
                    if src_png and src_png.is_file():
                        shutil.copy2(src_png, out_fp / norm / "qc" / png_name)
            elif use_qced and norm == "patch_normalized" and qced_patch_dir is not None:
                qced_png = qced_patch_dir / png_name
                if qced_png.is_file():
                    shutil.copy2(qced_png, out_fp / norm / "qc" / png_name)
                else:
                    src_png = Path(data[path_key_png]) if data.get(path_key_png) else None
                    if src_png and src_png.is_file():
                        shutil.copy2(src_png, out_fp / norm / "qc" / png_name)
            elif use_qced and norm == "patch_normalized":
                src_png = Path(data[path_key_png]) if data.get(path_key_png) else None
                if src_png and src_png.is_file():
                    shutil.copy2(src_png, out_fp / norm / "qc" / png_name)
            else:
                src_png = Path(data[path_key_png]) if data.get(path_key_png) else None
                if src_png and src_png.is_file():
                    shutil.copy2(src_png, out_fp / norm / "qc" / png_name)
            if src_npy and src_npy.is_file():
                shutil.copy2(src_npy, out_fp / norm / npy_name)
        copied += 1
    if allowed_pngs is not None and skipped:
        print(f"  fp/{iid}: skipped {skipped} QC rows not in qced whitelist")
    return copied

In [37]:
# Copy dents from QC results (approved/edited only) into ml_data_v4/dent/{iid} with filename = {sample_id}_{final_labels}
for iid in INSPECTION_IDS:
    n = copy_dents_from_qc_json(iid)
    print(f"dent/{iid}: copied {n} samples (approved/edited) with final_labels in filename")

dent/0A1OLRV96N3: copied 20 samples (approved/edited) with final_labels in filename
dent/0A49KLT3B7Y: copied 23 samples (approved/edited) with final_labels in filename
dent/0AGBXB4WEGN: copied 441 samples (approved/edited) with final_labels in filename
dent/0ABP0TFUSH1: copied 1277 samples (approved/edited) with final_labels in filename


In [42]:
# Copy FP into ml_data_v4/fp/{iid}; 0ABP: whitelist from std .../qc/qced, PNGs copied from qced dirs
for iid in INSPECTION_IDS:
    n = copy_fp_from_qc_json(iid)
    if n > 0:
        print(f"fp/{iid}: copied {n} samples (all QC rows passing qced whitelist when applicable)")

FP whitelist: /home/zmirikha/Github/rnd_q2/data/0ABP0TFUSH1/False Positive/std_normalized/qc/qced (80 PNGs); patch qced: None
  fp/0ABP0TFUSH1: skipped 8470 QC rows not in qced whitelist
fp/0ABP0TFUSH1: copied 76 samples (all QC rows passing qced whitelist when applicable)


## Gaussian noise & intensity scaling — **moved**

This step is now in **`augmentation/augment_v4.ipynb`** (noise + scale on `ml_data_v4/pos/` only, after dent→pos roll augmentation). The cell below is a no-op reminder.

In [39]:
# Noise + scale augmentation moved to data_preparation/augmentation/augment_v4.ipynb
# (run after roll aug; writes under ml_data_v4/pos/ only).
print(
    "Skip: use augment_v4.ipynb → augment_pos_noise_and_scale(AUG_OUT) for noise/scale on pos/."
)

Skip: use augment_v4.ipynb → augment_pos_noise_and_scale(AUG_OUT) for noise/scale on pos/.


In [40]:
# Summary
print("ml_data_v4 layout:")
for category in ["dent", "girth_weld", "random", "fp"]:
    cat_path = OUT_ROOT / category
    if cat_path.exists():
        for iid_path in sorted(cat_path.iterdir()):
            if iid_path.is_dir():
                for norm in ["std_normalized", "patch_normalized"]:
                    qc = iid_path / norm / "qc"
                    npy_dir = iid_path / norm
                    n_png = len(list(qc.glob("*.png"))) if qc.exists() else 0
                    n_npy = len(list(npy_dir.glob("*.npy"))) if npy_dir.exists() else 0
                    print(f"  {category}/{iid_path.name}/{norm}: {n_png} PNGs in qc/, {n_npy} NPYs")

ml_data_v4 layout:
  dent/0A1OLRV96N3/std_normalized: 20 PNGs in qc/, 20 NPYs
  dent/0A1OLRV96N3/patch_normalized: 20 PNGs in qc/, 20 NPYs
  dent/0A49KLT3B7Y/std_normalized: 23 PNGs in qc/, 23 NPYs
  dent/0A49KLT3B7Y/patch_normalized: 23 PNGs in qc/, 23 NPYs
  dent/0ABP0TFUSH1/std_normalized: 1178 PNGs in qc/, 1277 NPYs
  dent/0ABP0TFUSH1/patch_normalized: 1277 PNGs in qc/, 1277 NPYs
  dent/0AGBXB4WEGN/std_normalized: 441 PNGs in qc/, 441 NPYs
  dent/0AGBXB4WEGN/patch_normalized: 441 PNGs in qc/, 441 NPYs
  girth_weld/0A1OLRV96N3/std_normalized: 500 PNGs in qc/, 500 NPYs
  girth_weld/0A1OLRV96N3/patch_normalized: 500 PNGs in qc/, 500 NPYs
  girth_weld/0A49KLT3B7Y/std_normalized: 500 PNGs in qc/, 500 NPYs
  girth_weld/0A49KLT3B7Y/patch_normalized: 500 PNGs in qc/, 500 NPYs
  girth_weld/0ABP0TFUSH1/std_normalized: 1000 PNGs in qc/, 1000 NPYs
  girth_weld/0ABP0TFUSH1/patch_normalized: 1000 PNGs in qc/, 1000 NPYs
  girth_weld/0AGBXB4WEGN/std_normalized: 995 PNGs in qc/, 995 NPYs
  girth_we